In [ ]:
import logging
import os
import re
import time

import boto3

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)


def required_setting(name):
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Set the {name} environment variable")
    return value


# AWS credentials are resolved by boto3 from the standard provider chain.
glue_db = os.getenv("GLUE_DATABASE", "aw_db")
redshift_cluster_id = required_setting("REDSHIFT_CLUSTER_ID")
redshift_db = os.getenv("REDSHIFT_DATABASE", "dev")
redshift_user = required_setting("REDSHIFT_USER")
redshift_schema = os.getenv("REDSHIFT_SCHEMA", "public")
region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")

glue = boto3.client("glue", region_name=region)
redshift = boto3.client("redshift-data", region_name=region)
logger.info("Preparing Redshift tables from Glue database %s", glue_db)

In [ ]:
# --- Glue to Redshift Type Mapper ---
# Converts Glue data types to compatible Redshift data types
def map_glue_to_redshift(glue_type):
    mapping = {
        "string": "VARCHAR(256)",    # Default string size — tune based on your workload
        "int": "INTEGER",
        "bigint": "BIGINT",
        "double": "FLOAT8",
        "float": "FLOAT4",
        "boolean": "BOOLEAN",
        "timestamp": "TIMESTAMP",
        "date": "DATE"
    }
    # Fallback to VARCHAR for any unknown type to avoid breaking DDL
    return mapping.get(glue_type.lower(), "VARCHAR(256)")



In [ ]:
# --- Retrieve All Tables from Glue Catalog ---
# Uses pagination to fetch metadata for all tables under the specified Glue database
paginator = glue.get_paginator("get_tables")
pages = paginator.paginate(DatabaseName=glue_db)

all_tables = []
for page in pages:
    for table in page["TableList"]:
        all_tables.append(table["Name"])

print(f"Found {len(all_tables)} tables in Glue database '{glue_db}'")



In [ ]:
# Create Redshift tables for all tables in the configured Glue database.
for table in all_tables:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", table) or not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*", redshift_schema
    ):
        raise ValueError(f"Unsupported SQL identifier: {redshift_schema}.{table}")

    try:
        response = glue.get_table(DatabaseName=glue_db, Name=table)
        columns = response["Table"]["StorageDescriptor"]["Columns"]
        if not columns:
            raise ValueError(f"Glue table {table} has no columns")

        ddl = f"CREATE TABLE IF NOT EXISTS {redshift_schema}.{table} (\n"
        ddl += ",\n".join(
            f"  {column['Name']} {map_glue_to_redshift(column['Type'])}"
            for column in columns
        )
        ddl += "\n);"

        result = redshift.execute_statement(
            ClusterIdentifier=redshift_cluster_id,
            Database=redshift_db,
            DbUser=redshift_user,
            Sql=ddl
        )
        statement_id = result["Id"]
        while True:
            status = redshift.describe_statement(Id=statement_id)
            if status["Status"] == "FINISHED":
                logger.info("Created or verified Redshift table %s.%s", redshift_schema, table)
                break
            if status["Status"] in ("FAILED", "ABORTED", "CANCELED"):
                raise RuntimeError(
                    f"Redshift DDL failed for {table}: {status.get('Error', status['Status'])}"
                )
            time.sleep(2)
    except Exception:
        logger.exception("Failed to create Redshift table %s.%s", redshift_schema, table)
        raise